In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import numpy as np
import config
import os
import pickle
import torch
import tqdm

from src.metric import *
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.cross_encoder import CrossEncoder

from src.bi_encoder_training import get_resume_embedding,get_jd_embedding


In [3]:
path=config.CLEANED_DATA_DIR
BATCH_SIZE=config.BATCH_SIZE

In [4]:
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)

In [5]:
bi_encoder=SentenceTransformer(os.path.join(config.CHUNKED_MODEL_DIR,'bi_encoder_chunked'))

cross_encoder=CrossEncoder(os.path.join(config.CHUNKED_MODEL_DIR,'cross_encoder_chunked'),num_labels=1)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [6]:
bi_encoder_hn=SentenceTransformer(os.path.join(config.HD_MODEL_DIR,'bi_encoder_hard_negatives'))

cross_encoder_hn = CrossEncoder(os.path.join(config.HD_MODEL_DIR,'cross_encoder_hard_negatives'))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [7]:
test_resume_texts=test_df['resume_text'].drop_duplicates().tolist()

In [8]:
bi_encoder.eval()

test_resume_embs=[]
with torch.no_grad():
  for resume in tqdm.tqdm(test_resume_texts):
    emb=get_resume_embedding(bi_encoder,resume)
    test_resume_embs.append(emb)
    del emb


100%|██████████| 477/477 [00:08<00:00, 59.16it/s]


In [9]:
def retrieve_candidates(model,jd,label_map,test_resume_embs,test_resume_texts,top_k=50):

    jd_emb=get_jd_embedding(model,jd)

    hits=util.semantic_search(jd_emb,test_resume_embs,top_k=top_k)
    hits=hits[0]

    retrieval_rows=[]

    for hit in hits:

        idx=hit["corpus_id"]

        resume_text=test_resume_texts[idx]

        if resume_text not in label_map:
            continue

        retrieval_rows.append({
            "resume_text": resume_text,
            "score": float(hit["score"]),
            "label": label_map[resume_text]
        })

    return pd.DataFrame(retrieval_rows)

In [10]:
def rerank_candidates(model,jd,retrieval_df):

    pairs=list(zip([jd]*len(retrieval_df),retrieval_df['resume_text']))

    cross_scores=model.predict(pairs,batch_size=BATCH_SIZE)

    rerank_df=retrieval_df.copy()

    rerank_df["score"]=cross_scores

    rerank_df=rerank_df.sort_values("score",ascending=False).reset_index(drop=True)

    return rerank_df

In [11]:
def evaluate_df(df):
    metrics={}

    metrics["ndcg"]=ndcg_metric(df)

    metrics["spearman"]=corr_metric(df)

    metrics["topk"]=topk_metric(df)

    metrics["mrr"]=mrr_metric(df)

    metrics["map"]=map_metric(df)

    return metrics

In [12]:
def initialize_metrics():
    return {"ndcg": [],"spearman": [],"topk": [],"map": [],"mrr": [] }
      

In [15]:
def two_stage_pipeline(df,retrieval_model,reranker_model,resume_embs,resume_texts):
    
    bi_encoder_metrics=initialize_metrics()
    cross_encoder_metrics=initialize_metrics()

    for jd, group in df.groupby("job_description_text"):

        if len(group)<2:
            continue

        group_label=group[['resume_text','label']].drop_duplicates()
        label_map=dict(zip(group_label['resume_text'],group_label['label']))

        # Stage 1 Retrieval

        retrieval_df=retrieve_candidates(retrieval_model,jd,label_map,resume_embs,resume_texts,top_k=50)

        if retrieval_df.empty:
            continue

        bi_metrics=evaluate_df(retrieval_df)

        for key in bi_encoder_metrics:
            if bi_metrics[key] is not None:
                bi_encoder_metrics[key].append(bi_metrics[key])

        # Stage 2 Reranking

        rerank_df=rerank_candidates(reranker_model,jd,retrieval_df)

        cross_metrics=evaluate_df(rerank_df)

        for key in cross_encoder_metrics:
            if cross_metrics[key] is not None:
                cross_encoder_metrics[key].append(cross_metrics[key])

    return bi_encoder_metrics,cross_encoder_metrics

In [16]:
bi_encoder_metrics,cross_encoder_metrics=two_stage_pipeline(test_df,bi_encoder,cross_encoder,test_resume_embs,test_resume_texts)

In [17]:
bi_encoder_hn_metrics,cross_encoder_hn_metrics=two_stage_pipeline(test_df,bi_encoder_hn,cross_encoder_hn,test_resume_embs,test_resume_texts)

In [18]:
def summarize_metrics(bi_encoder_metrics,cross_encoder_metrics):
    results_df=pd.DataFrame({
        "metric": list(bi_encoder_metrics.keys()),
        "bi_encoder": [np.mean(vals) if vals else 0 for vals in bi_encoder_metrics.values()],
        "cross_encoder": [np.mean(vals) if vals else 0 for vals in cross_encoder_metrics.values()]
    })
    return results_df

In [19]:
results_df=summarize_metrics(bi_encoder_metrics,cross_encoder_metrics)
hn_results_df=summarize_metrics(bi_encoder_hn_metrics,cross_encoder_hn_metrics)

In [20]:
print("\n" + "="*60)
print("FINAL TWO-STAGE RETRIEVAL RESULTS WITHOUT HARD NEGATIVES")
print("="*60)

print(f"{'Metric':<20}{'Bi-Encoder':>10}{'Cross-Encoder':>15}")
for _, row in results_df.iterrows():
    print(
        f"{row['metric']:<20}"
        f"{row['bi_encoder']:>10.4f}"
        f"{row['cross_encoder']:>15.4f}"
    )

print("="*60)


FINAL TWO-STAGE RETRIEVAL RESULTS WITHOUT HARD NEGATIVES
Metric              Bi-Encoder  Cross-Encoder
ndcg                    0.8530         0.8474
spearman                0.1263         0.0495
topk                    1.0000         0.8421
map                     0.8703         0.8765
mrr                     0.8958         0.9246


In [21]:
print("\n" + "="*60)
print("FINAL TWO-STAGE RETRIEVAL RESULTS WITH HARD NEGATIVES")
print("="*60)

print(f"{'Metric':<20}{'Bi-Encoder':>10}{'Cross-Encoder':>15}")
for _, row in hn_results_df.iterrows():
    print(
        f"{row['metric']:<20}"
        f"{row['bi_encoder']:>10.4f}"
        f"{row['cross_encoder']:>15.4f}"
    )

print("="*60)


FINAL TWO-STAGE RETRIEVAL RESULTS WITH HARD NEGATIVES
Metric              Bi-Encoder  Cross-Encoder
ndcg                    0.7225         0.7476
spearman                0.0403         0.0450
topk                    0.8333         0.8750
map                     0.8205         0.8399
mrr                     0.8352         0.8646
